# Tekrarlayan Sinir Ağları (RNN & LSTM) ile Zaman Serisi Tahmini

Bu modül; sıralı verilerde (zaman serileri, sensör telemetrisi, metinler) geçmiş adımların bilgisini taşıyan **Tekrarlayan Sinir Ağları (Recurrent Neural Networks - RNN)** ve kaybolan gradyan (vanishing gradient) problemini çözen **Uzun Kısa Vadeli Hafıza (Long Short-Term Memory - LSTM)** mimarilerini inceler.

---

## 1. LSTM Hücresinin Matematiksel Anatomisi

Geleneksel RNN'ler uzun vadeli bağımlılıkları öğrenirken $\frac{\partial h_t}{\partial h_1}$ gradyanı sıfıra yaklaşarak kaybolur (Vanishing Gradient). Hochreiter & Schmidhuber (1997) tarafından önerilen LSTM hücresi, bilgiyi korumak için 3 kapı (gate) ve bir hücre durumu (cell state $C_t$) kullanır:

1. **Unutma Kapısı (Forget Gate):** Önceki hücre durumundan nelerin atılacağını belirler:
   $$f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

2. **Girdi Kapısı (Input Gate):** Hangi yeni bilgilerin hücre durumuna yazılacağını seçer:
   $$i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$$
   $$\tilde{C}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$

3. **Hücre Durumu Güncellemesi (Cell State Update):**
   $$C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$$

4. **Çıktı Kapısı (Output Gate):** Hücre durumunun hangi kısmının gizli duruma ($h_t$) aktarılacağını belirler:
   $$o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$$
   $$h_t = o_t \odot \tanh(C_t)$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Sentetik Karmaşık Zaman Serisi Verisi Üretimi (Trend + Mevsimsellik + Gürültü)
np.random.seed(42)
time_steps = 1000
time = np.arange(time_steps)
trend = 0.05 * time
seasonality = 10 * np.sin(2 * np.pi * time / 50) + 5 * np.cos(2 * np.pi * time / 25)
noise = np.random.normal(0, 1.5, size=time_steps)
series = trend + seasonality + noise

plt.figure(figsize=(12, 4))
plt.plot(time, series, label='Zaman Serisi (Gözlemler)', color='steelblue')
plt.title("Sentetik Karmaşık Zaman Serisi Verisi")
plt.xlabel("Zaman Adımı (t)")
plt.ylabel("Değer")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 2. Kayan Pencere (Sliding Window) Tensör Hazırlığı

LSTM modelleri girdileri 3 boyutlu tensör olarak bekler: `[örnek_sayısı, zaman_adımı, öznitelik_sayısı]`.


In [ ]:
# Veriyi [0, 1] aralığına ölçekleme
series_min = series.min()
series_max = series.max()
scaled_series = (series - series_min) / (series_max - series_min)

WINDOW_SIZE = 30  # Geçmiş 30 adıma bakarak bir sonraki adımı tahmin etme

X, y = [], []
for i in range(len(scaled_series) - WINDOW_SIZE):
    X.append(scaled_series[i:i + WINDOW_SIZE])
    y.append(scaled_series[i + WINDOW_SIZE])

X = np.array(X)[..., np.newaxis]  # Şekil: (N, 30, 1)
y = np.array(y)

# Train-Test Ayrımı (%80 Eğitim, %20 Test)
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"X_train Şekli: {X_train.shape}, y_train Şekli: {y_train.shape}")
print(f"X_test Şekli : {X_test.shape}, y_test Şekli : {y_test.shape}")


## 3. Keras LSTM Modelinin Kurulumu ve Eğitimi

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(WINDOW_SIZE, 1)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),
    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)  # Tekil sürekli çıktı
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.005),
    loss='mse',
    metrics=['mae']
)

model.summary()

history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_split=0.15,
    verbose=1
)


## 4. Test Tahminleri ve Başarım Görselleştirmesi

In [ ]:
# Test kümesinde tahmin alma
predictions = model.predict(X_test)

# Orijinal ölçeğe geri dönüştürme (Inverse Scale)
y_test_orig = y_test * (series_max - series_min) + series_min
predictions_orig = predictions.squeeze() * (series_max - series_min) + series_min

# MAE ve RMSE hesabı
mae = np.mean(np.abs(y_test_orig - predictions_orig))
rmse = np.sqrt(np.mean((y_test_orig - predictions_orig) ** 2))

print(f"Test MAE : {mae:.3f}")
print(f"Test RMSE: {rmse:.3f}")

plt.figure(figsize=(14, 5))
plt.plot(y_test_orig, label='Gerçek Değerler', color='black', alpha=0.7)
plt.plot(predictions_orig, label='LSTM Tahminleri', color='crimson', linestyle='--')
plt.title(f"LSTM Zaman Serisi Test Tahmini (MAE: {mae:.2f}, RMSE: {rmse:.2f})")
plt.xlabel("Test Zaman Adımları")
plt.ylabel("Değer")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
